# 01 — Data Collection

Three sources combined:

| Source | Data | Years |
|---|---|---|
| **CollegeFootballData API** | Game results (scores, W/L) | 2005–2024 |
| **School athletic sites** | Rosters (height, weight, position) | 2014–2024 |
| **Sports-Reference CSV exports** | Season stats (yards, pass rate) | manual |


In [ ]:
import sys
sys.path.insert(0, '..')
from src.scraper import IVY_CFBD_NAMES
from src.roster_scraper import SCHOOL_SITES
print('CFBD schools:', list(IVY_CFBD_NAMES.keys()))
print('Roster sites:', list(SCHOOL_SITES.keys()))

## Step 1 — Game results via CollegeFootballData API

In [ ]:
from src.scraper import scrape_all
scrape_all(start_year=2005, end_year=2024, data_dir='../data/raw')

## Step 2 — Rosters from school athletic sites

Scrapes all 8 schools 2014–2024. Uses Sidearm/Nuxt parser for Brown, Columbia, Dartmouth, Penn, Princeton and HTML table parser for Harvard, Yale, Cornell. Note: Harvard does not publish player weights.

In [ ]:
from src.roster_scraper import scrape_all_rosters
rosters = scrape_all_rosters(start_year=2014, end_year=2024, data_dir='../data/raw')
print(rosters.groupby(['school','year']).size().reset_index(name='players').to_string())

## Step 3 — Season stats derived from game results

In [ ]:
import pandas as pd
from src.features import build_stats_from_games

schedules = pd.read_csv('../data/raw/schedules/schedules_raw.csv')
stats = build_stats_from_games(schedules)
stats.to_csv('../data/raw/team_stats/team_stats_raw.csv', index=False)
print('Stats:', stats.shape)
stats.head(10)

## Step 4 — (Optional) Sports-Reference manual CSV import

1. Go to `https://www.sports-reference.com/cfb/schools/{school}/{year}.html`
2. Click the **CSV** button above the Team Stats table
3. Save as `data/raw/sr_exports/{school}_{year}.csv`
4. Run this cell to merge


In [ ]:
from pathlib import Path
import pandas as pd

sr_dir = Path('../data/raw/sr_exports')
sr_dir.mkdir(exist_ok=True)
sr_files = list(sr_dir.glob('*.csv'))

if not sr_files:
    print('No S-R exports found — place CSVs in data/raw/sr_exports/ as school_year.csv')
else:
    frames = []
    for f in sr_files:
        parts = f.stem.split('_')
        school, year = '_'.join(parts[:-1]), int(parts[-1])
        df = pd.read_csv(f, comment='#')
        df['school'], df['year'] = school, year
        frames.append(df)
    sr_stats = pd.concat(frames, ignore_index=True)
    sr_stats.to_csv('../data/raw/team_stats/sr_stats_raw.csv', index=False)
    print(f'Saved {len(sr_stats)} rows from {len(sr_files)} exports')
    display(sr_stats.head())

## Summary

In [ ]:
import pandas as pd

rosters   = pd.read_csv('../data/raw/rosters/rosters_raw.csv')
schedules = pd.read_csv('../data/raw/schedules/schedules_raw.csv')
stats     = pd.read_csv('../data/raw/team_stats/team_stats_raw.csv')

print(f'Games  : {len(schedules):,} rows | {schedules.school.nunique()} schools | {schedules.year.min()}–{schedules.year.max()}')
print(f'Rosters: {len(rosters):,} rows  | {rosters.school.nunique()} schools | {rosters.year.min()}–{rosters.year.max()}')
print(f'Stats  : {len(stats):,} rows  | derived from game results')
print()
display(stats.groupby('school')['year'].agg(['min','max','count']))
print()
display(rosters.groupby(['school','year']).size().reset_index(name='players').pivot(index='school',columns='year',values='players').fillna(0).astype(int))